# NACA Aerodynamic Surrogate — Training Analysis

End-to-end walkthrough: data, training, evaluation, and externally scoped reference comparison.

Run cells sequentially after completing:
1. `python -m data.generate_dataset --out data/dataset_v2.csv`
2. `python -m src.train --data data/dataset_v2.csv`

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch

sys.path.insert(0, str(Path.cwd().parent))

plt.rcParams.update({
    'figure.facecolor': '#0e1420',
    'axes.facecolor': '#080c14',
    'axes.edgecolor': '#2a3a50',
    'axes.labelcolor': '#8090b0',
    'text.color': '#c8d4e8',
    'xtick.color': '#5a6a82',
    'ytick.color': '#5a6a82',
    'grid.color': '#151e2e',
    'grid.linestyle': '--',
    'grid.alpha': 0.5,
    'lines.linewidth': 2,
})
print('Imports OK')

## 1. Dataset Exploration

In [ ]:
df = pd.read_csv('../data/dataset_v2.csv')
print(f'{len(df):,} samples, columns: {list(df.columns)}')
df[['Cl','Cd','LD']].describe().round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Output distributions', color='#c8d4e8', fontsize=12)
for ax, col, color in zip(axes, ['Cl','Cd','LD'], ['#6ab4f5','#e87030','#50d880']):
    ax.hist(df[col], bins=80, color=color, alpha=0.8, edgecolor='none')
    ax.set_xlabel(col); ax.set_ylabel('count')
    ax.grid(True)
plt.tight_layout(); plt.show()

## 2. Training Loss Curves

In [ ]:
ckpt = torch.load('../model/surrogate_model.pt', map_location='cpu')
hist = ckpt['history']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Training convergence', color='#c8d4e8')

ax = axes[0]
ax.plot(hist['train_data'], label='Train data loss', color='#6ab4f5')
ax.plot(hist['val'],        label='Val loss',        color='#50d880')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE (normalised)'); ax.legend(); ax.grid(True)

ax = axes[1]
ax.plot(hist['train_phys'], label='Physics loss', color='#e87030')
ax.set_xlabel('Epoch'); ax.set_ylabel('Physics MSE'); ax.legend(); ax.grid(True)

plt.tight_layout(); plt.show()
print(f'Best val loss: {min(hist["val"]):.5f}')
print(f'Test R²: Cl={ckpt["test_r2"][0]:.4f}  Cd={ckpt["test_r2"][1]:.4f}')

## 3. Model Evaluation — Cl and Cd Polars

In [ ]:
from src.model import AeroSurrogate
from src.thin_airfoil_theory import lift_coefficient_batch
from src.drag_model import drag_coefficient_batch
from src.airfoil_geometry import zero_lift_angle_batch
from src.thin_airfoil_theory import stall_angle_batch

model = AeroSurrogate()
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

in_mean = np.array(ckpt['input_mean'],  dtype=np.float32)
in_std  = np.array(ckpt['input_std'],   dtype=np.float32)
out_mean= np.array(ckpt['output_mean'], dtype=np.float32)
out_std = np.array(ckpt['output_std'],  dtype=np.float32)

def predict(m, p, t, alphas, Re):
    xs = np.column_stack([
        np.full_like(alphas, m),
        np.full_like(alphas, p),
        np.full_like(alphas, t),
        alphas,
        np.full_like(alphas, np.log10(Re))
    ]).astype(np.float32)
    xn = (xs - in_mean) / (in_std + 1e-8)
    with torch.no_grad():
        yn = model(torch.from_numpy(xn)).numpy()
    y = yn * out_std + out_mean
    return y[:, 0], y[:, 1]  # Cl, Cd

alphas = np.linspace(-10, 20, 240)
# NACA 2412
cl_nn, cd_nn = predict(0.02, 0.4, 0.12, alphas, 1e6)
a0 = zero_lift_angle_batch(np.array([0.02]), np.array([0.4]))
cl_th = 2 * np.pi * (np.radians(alphas) - a0[0])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('NACA 2412  ·  Re = 1×10⁶', color='#c8d4e8')

axes[0].plot(alphas, cl_th, '--', color='#e8a040', label='Thin airfoil theory')
axes[0].plot(alphas, cl_nn, color='#6ab4f5', label='NN surrogate')
axes[0].set_xlabel('α (deg)'); axes[0].set_ylabel('Cl'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(alphas, cd_nn, color='#e87030')
axes[1].set_xlabel('α (deg)'); axes[1].set_ylabel('Cd'); axes[1].grid(True)

axes[2].plot(cd_nn, cl_nn, color='#a060f8')
axes[2].set_xlabel('Cd'); axes[2].set_ylabel('Cl')
axes[2].set_title('Drag polar'); axes[2].grid(True)

plt.tight_layout(); plt.show()

## 4. Approximate and external reference comparison

In [ ]:
from data.uiuc_loader import get_validation_data

for profile, m, p, t in [('naca2412',0.02,0.4,0.12), ('naca0012',0,0.4,0.12)]:
    uiuc = get_validation_data(profile, Re=1e6)
    if uiuc.empty: continue
    
    cl_nn_v, cd_nn_v = predict(m, p, t, np.array(uiuc['alpha'].values, dtype=np.float32), 1e6)
    
    def r2_score(y_true, y_pred):
        y_true = np.asarray(y_true, dtype=float)
        y_pred = np.asarray(y_pred, dtype=float)
        return 1.0 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - y_true.mean()) ** 2)

    r2_cl = r2_score(uiuc['Cl'], cl_nn_v)
    r2_cd = r2_score(uiuc['Cd'], cd_nn_v)
    
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    fig.suptitle(f'{profile.upper()}  ·  Re=1e6  ·  R²: Cl={r2_cl:.3f}  Cd={r2_cd:.3f}', color='#c8d4e8')
    
    axes[0].scatter(uiuc['alpha'], uiuc['Cl'], s=25, color='#50d880', label='Approximate reference', zorder=3)
    axes[0].plot(uiuc['alpha'], cl_nn_v, color='#6ab4f5', label='NN', linewidth=2)
    axes[0].set_xlabel('α (deg)'); axes[0].set_ylabel('Cl'); axes[0].legend(); axes[0].grid(True)
    
    axes[1].scatter(uiuc['alpha'], uiuc['Cd'], s=25, color='#50d880', label='Approximate reference', zorder=3)
    axes[1].plot(uiuc['alpha'], cd_nn_v, color='#e87030', label='NN', linewidth=2)
    axes[1].set_xlabel('α (deg)'); axes[1].set_ylabel('Cd'); axes[1].legend(); axes[1].grid(True)
    
    plt.tight_layout(); plt.show()

## 5. Physics Constraint Verification

In [ ]:
# Verify symmetry constraint: NACA 0012 at alpha=0 should give Cl=0
for m, p, t, label in [(0.0, 0.4, 0.12, 'NACA 0012'), (0.0, 0.2, 0.06, 'NACA 0006')]:
    cl_at_zero, _ = predict(m, p, t, np.array([0.0]), 1e6)
    print(f'{label:10s}  Cl(alpha=0) = {cl_at_zero[0]:.5f}  (should be ~0)')

# Verify zero-lift angle: Cl should be ~0 at alpha=alpha_0
from src.airfoil_geometry import zero_lift_angle
for m, p, t, label in [(0.02,0.4,0.12,'NACA 2412'), (0.04,0.4,0.12,'NACA 4412')]:
    a0 = np.degrees(zero_lift_angle(m, p))
    cl_at_a0, _ = predict(m, p, t, np.array([a0], dtype=np.float32), 1e6)
    print(f'{label:10s}  Cl(alpha_0={a0:.2f}°) = {cl_at_a0[0]:.5f}  (should be ~0)')

## 6. Design Space — L/D Heatmap (Re=1×10⁶, α=5°)

In [ ]:
m_vals = np.linspace(0, 9, 30) / 100
t_vals = np.linspace(6, 24, 30) / 100
LD_map = np.zeros((len(m_vals), len(t_vals)))

for i, mv in enumerate(m_vals):
    for j, tv in enumerate(t_vals):
        cl, cd = predict(mv, 0.4, tv, np.array([5.0]), 1e6)
        LD_map[i, j] = cl[0] / cd[0] if cd[0] > 0 else 0

fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(LD_map, origin='lower', aspect='auto',
               extent=[6, 24, 0, 9], cmap='plasma', vmin=0, vmax=120)
plt.colorbar(im, ax=ax, label='L/D')
ax.set_xlabel('Max thickness t (%)'); ax.set_ylabel('Max camber m (%)')
ax.set_title('Design space: L/D  ·  α=5°  ·  Re=1×10⁶', color='#c8d4e8')
plt.tight_layout(); plt.show()

## 7. Export to Dashboard

```bash
# After this notebook is complete:
python -m src.export_weights --model model/surrogate_model.pt \
                              --output model/model_weights.json \
                              --embed  dashboard/index.html
# Then open dashboard/index.html in a browser.
```